# ⚡ match_dos_v2 — Notebook OPTIMIZADO para RAM

Basado en `match_mirai_v2_optimizado.ipynb`. Aplica desde la celda 0 todas las lecciones aprendidas con Mirai:
- `import gc` y liberación explícita de sets/columnas intermedias.
- Sampling configurable de Tshark.
- Offset horario parametrizado (`OFFSET_HORAS`) — comprobar si DoS es invierno (1h) o verano (2h).
- Sin `reconstruir_ip` (particularidad sólo de Mirai).
- Etiqueta final colapsada a `'DoS'` para los 4 subtipos (TCP, UDP, SYN, HTTP).

# ============================================================
# NOTEBOOK: match_dos_v2.ipynb
# DoS — Matching entre CIC, Nemea, Tstat y Tshark
# ============================================================
#
# DISTRIBUCIÓN DoS:
#   - CIC:    /ArchivosCIC/DoS                  → 5 CSVs (1 TCP + 1 UDP + 1 SYN + 2 HTTP)
#   - Nemea:  /ArchivosNEMEA/DoS/DoS            → 4 CSVs (uno por subtipo)
#   - Tstat:  /ArchivosTstat/CSV/DoS            → 4 CSVs (uno por subtipo)
#   - Tshark: /ArchivosTshark_2/Csv/DoS         → 4 CSVs (uno por subtipo)
#
# OBJETIVO:
#   Superar la minoritaria actual (Web, 3.481 flujos en Comunes 4).
#
# COSAS A VIGILAR EN LA PRIMERA EJECUCIÓN:
#   1. Comprobar offset horario en celda 2.2 (1h invierno vs 2h verano).
#   2. Si Tshark satura RAM, bajar N_SAMPLE en celda 4.1.

In [1]:
# ============================================================
# CELDA 0 — IMPORTS GLOBALES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import gc
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── PARÁMETROS GLOBALES ───────────────────────────────────────
OFFSET_HORAS = 1          # 1=invierno (CET), 2=verano (CEST). Verificar con Mirai (era 1).
N_SAMPLE_TSHARK = 50_000  # filas máx por CSV de Tshark (sampling para no petar RAM)

print('✓ Imports OK')
print(f'  pandas:          {pd.__version__}')
print(f'  numpy:           {np.__version__}')
print(f'  OFFSET_HORAS:    {OFFSET_HORAS}h')
print(f'  N_SAMPLE_TSHARK: {N_SAMPLE_TSHARK:,}')

✓ Imports OK
  pandas:          2.2.0
  numpy:           1.26.4
  OFFSET_HORAS:    1h
  N_SAMPLE_TSHARK: 50,000


# CICFlowMeter

In [2]:
# ============================================================
# CELDA 1.1 — CARGA DE CIC
# DoS tiene 5 CSVs (TCP, UDP, SYN, HTTP×2) → glob
# ============================================================

RUTA_CIC = Path('/home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosCIC/DoS')
print(f'Ruta CIC: {RUTA_CIC}')
print(f'Existe:   {RUTA_CIC.exists()}')
print()

csv_files = sorted(RUTA_CIC.glob('*.csv'))
print(f'CSVs encontrados: {len(csv_files)}')

if not csv_files:
    raise FileNotFoundError(f'No se encontraron CSVs en: {RUTA_CIC}')

dfs = []
for f in csv_files:
    df_temp = pd.read_csv(f, low_memory=False)
    df_temp['archivo_origen'] = f.stem
    dfs.append(df_temp)
    print(f'  ✓ {f.name:50s} → {len(df_temp):>8,} filas')

df_cic = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print()
print(f'Shape total CIC: {df_cic.shape}')
print(f'NaNs totales:    {df_cic.isnull().sum().sum():,}')

Ruta CIC: /home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosCIC/DoS
Existe:   True

CSVs encontrados: 5
  ✓ DoS-HTTP_Flood.pcap_Flow.csv                       →  932,513 filas
  ✓ DoS-HTTP_Flood1.pcap_Flow.csv                      →  710,231 filas
  ✓ DoS-SYN_Flood.pcap_Flow.csv                        → 3,990,181 filas
  ✓ DoS-TCP_Flood.pcap_Flow.csv                        → 1,637,349 filas
  ✓ DoS-UDP_Flood.pcap_Flow.csv                        →  934,001 filas

Shape total CIC: (8204275, 85)
NaNs totales:    79,836


In [3]:
# ============================================================
# CELDA 1.2 — LIMPIEZA Y NORMALIZACIÓN CIC
# ============================================================

df_cic['Timestamp'] = (
    df_cic['Timestamp'].astype(str)
    .str.replace(' p. m.', ' PM', regex=False)
    .str.replace(' a. m.', ' AM', regex=False)
    .str.strip()
)
df_cic['Timestamp'] = pd.to_datetime(
    df_cic['Timestamp'], format='%d/%m/%Y %I:%M:%S %p', errors='coerce'
)

nat_count = df_cic['Timestamp'].isna().sum()
print(f'NaT en Timestamp: {nat_count}')
if nat_count > 0:
    df_cic = df_cic.dropna(subset=['Timestamp']).reset_index(drop=True)

df_cic['Protocol']      = pd.to_numeric(df_cic['Protocol'],      errors='coerce').fillna(0).astype(int)
df_cic['Src Port']      = pd.to_numeric(df_cic['Src Port'],      errors='coerce').fillna(0).astype(int)
df_cic['Dst Port']      = pd.to_numeric(df_cic['Dst Port'],      errors='coerce').fillna(0).astype(int)
df_cic['Flow Duration'] = pd.to_numeric(df_cic['Flow Duration'], errors='coerce')

print(f'\nShape CIC limpio: {df_cic.shape}')
print(f'\nRango temporal CIC:')
print(f'  Min: {df_cic["Timestamp"].min()}')
print(f'  Max: {df_cic["Timestamp"].max()}')
print(f'\nDistribución por protocolo:')
print(df_cic['Protocol'].value_counts().to_string())
print(f'\nDistribución por subtipo (archivo origen):')
print(df_cic['archivo_origen'].value_counts().to_string())

NaT en Timestamp: 0

Shape CIC limpio: (8204275, 85)

Rango temporal CIC:
  Min: 2022-07-22 16:41:31
  Max: 2022-08-10 15:19:33

Distribución por protocolo:
Protocol
6     7239632
17     963061
0        1582

Distribución por subtipo (archivo origen):
archivo_origen
DoS-SYN_Flood.pcap_Flow      3990181
DoS-TCP_Flood.pcap_Flow      1637349
DoS-UDP_Flood.pcap_Flow       934001
DoS-HTTP_Flood.pcap_Flow      932513
DoS-HTTP_Flood1.pcap_Flow     710231


In [4]:
# ============================================================
# CELDA 1.3 — KEY BASE Y KEY EXACT CIC
# ============================================================

df_cic['time_sec']    = df_cic['Timestamp'].dt.floor('s')
df_cic['flow_dur_ms'] = (df_cic['Flow Duration'] / 1000).round().astype('Int64')

df_cic['key_base'] = list(zip(
    df_cic['Src IP'].astype(str),
    df_cic['Src Port'].astype(str),
    df_cic['Dst IP'].astype(str),
    df_cic['Dst Port'].astype(str),
    df_cic['Protocol'].astype(str),
    df_cic['time_sec'].astype(str)
))
df_cic['key_exact'] = list(zip(
    df_cic['Src IP'].astype(str),
    df_cic['Src Port'].astype(str),
    df_cic['Dst IP'].astype(str),
    df_cic['Dst Port'].astype(str),
    df_cic['Protocol'].astype(str),
    df_cic['time_sec'].astype(str),
    df_cic['flow_dur_ms'].astype(str)
))

print(f'Shape CIC: {df_cic.shape}')
print(f'Claves base únicas: {df_cic["key_base"].nunique():,}')

Shape CIC: (8204275, 89)
Claves base únicas: 8,078,797


In [5]:
# ============================================================
# CELDA 1.4 — ANÁLISIS DE COLISIONES CIC
# ============================================================

conteo          = df_cic.groupby('key_base').size()
total_claves    = len(conteo)
claves_unicas   = (conteo == 1).sum()
claves_colision = (conteo > 1).sum()
flujos_colision = conteo[conteo > 1].sum()
pct_colision    = round(claves_colision / total_claves * 100, 2)

print('=' * 60)
print('ANÁLISIS DE COLISIONES CIC — DoS')
print('=' * 60)
print(f'  Total filas:                 {len(df_cic):,}')
print(f'  Total claves base únicas:    {total_claves:,}')
print(f'  Claves sin colisión:         {claves_unicas:,}  ({round(claves_unicas/total_claves*100,2)}%)')
print(f'  Claves con colisión:         {claves_colision:,}  ({pct_colision}%)')
print(f'  Flujos afectados:            {flujos_colision:,}')
print()
if pct_colision < 1:
    print(f'  → ✓ SEGURO ({pct_colision}% colisiones)')
elif pct_colision < 5:
    print(f'  → ~ ACEPTABLE ({pct_colision}% colisiones)')
else:
    print(f'  → ✗ RIESGO ({pct_colision}% colisiones)')

del conteo
gc.collect()

ANÁLISIS DE COLISIONES CIC — DoS
  Total filas:                 8,204,275
  Total claves base únicas:    8,078,797
  Claves sin colisión:         8,012,621  (99.18%)
  Claves con colisión:         66,176  (0.82%)
  Flujos afectados:            191,654

  → ✓ SEGURO (0.82% colisiones)


0

# NEMEA

In [ ]:
# ============================================================
# CELDA 2.1 — CARGA DE NEMEA
# DoS: 4 CSVs (uno por subtipo: TCP, UDP, SYN, HTTP) → glob
# Columna de etiqueta detectada dinámicamente (última columna)
# ============================================================

RUTA_NEMEA = Path('/home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosNEMEA/DoS/DoS')
print(f'Ruta Nemea: {RUTA_NEMEA}')
print(f'Existe:     {RUTA_NEMEA.exists()}')
print()

csv_files_nemea = sorted(RUTA_NEMEA.glob('*_Nemea.csv'))
if not csv_files_nemea:
    # Si no encuentra con sufijo _Nemea, prueba con *.csv
    csv_files_nemea = sorted(RUTA_NEMEA.glob('*.csv'))

print(f'CSVs Nemea encontrados: {len(csv_files_nemea)}')
for f in csv_files_nemea:
    print(f'  {f.name}')

if not csv_files_nemea:
    raise FileNotFoundError(f'No se encontraron CSVs Nemea en: {RUTA_NEMEA}')

# Detectar columna de etiqueta desde la cabecera del primer fichero
col_etiqueta_nemea = pd.read_csv(csv_files_nemea[0], nrows=0).columns[-1]
print(f'\nColumna de etiqueta detectada: \'{col_etiqueta_nemea}\'')

dfs_nemea = []
for f in csv_files_nemea:
    df_temp = pd.read_csv(f, low_memory=False)
    df_temp['archivo_origen_nemea'] = f.stem
    dfs_nemea.append(df_temp)
    print(f'  ✓ {f.name:45s} → {len(df_temp):>8,} filas')

df_nemea = pd.concat(dfs_nemea, ignore_index=True)
del dfs_nemea
gc.collect()

print()
print(f'Shape Nemea total: {df_nemea.shape}')
print(f'NaNs:              {df_nemea.isnull().sum().sum():,}')
print(f'\nDistribución por subtipo:')
print(df_nemea['archivo_origen_nemea'].value_counts().to_string())
print(f'\nDistribución por protocolo:')
print(df_nemea['uint8 PROTOCOL'].value_counts().to_string())
print(f'\nDistribución por etiqueta ({col_etiqueta_nemea}):')
print(df_nemea[col_etiqueta_nemea].value_counts().to_string())

In [ ]:
# ============================================================
# CELDA 2.2 — LIMPIEZA Y NORMALIZACIÓN NEMEA
# Sin reconstruir_ip (era exclusivo de Mirai).
# Offset configurable: OFFSET_HORAS (definido en CELDA 0).
# ============================================================

col_src_ip   = 'ipaddr SRC_IP'
col_dst_ip   = 'ipaddr DST_IP'
col_src_port = 'uint16 SRC_PORT'
col_dst_port = 'uint16 DST_PORT'
col_protocol = 'uint8 PROTOCOL'
col_t_first  = 'time TIME_FIRST'
col_t_last   = 'time TIME_LAST'

# Limpiar espacios en IPs
df_nemea[col_src_ip] = df_nemea[col_src_ip].astype(str).str.strip()
df_nemea[col_dst_ip] = df_nemea[col_dst_ip].astype(str).str.strip()

# Timestamps — coma decimal → punto (por si acaso, sin coste)
df_nemea['TIME_FIRST_limpio'] = df_nemea[col_t_first].astype(str).str.replace(',', '.', regex=False)
df_nemea['TIME_LAST_limpio']  = df_nemea[col_t_last].astype(str).str.replace(',', '.', regex=False)

df_nemea['TIME_FIRST_dt'] = pd.to_datetime(
    df_nemea['TIME_FIRST_limpio'], format='%Y-%m-%dT%H:%M:%S.%f', errors='coerce'
)
df_nemea['TIME_LAST_dt'] = pd.to_datetime(
    df_nemea['TIME_LAST_limpio'], format='%Y-%m-%dT%H:%M:%S.%f', errors='coerce'
)

# Aplicar offset horario configurable
df_nemea['TIME_FIRST_utc2'] = df_nemea['TIME_FIRST_dt'] + pd.Timedelta(hours=OFFSET_HORAS)
df_nemea['TIME_LAST_utc2']  = df_nemea['TIME_LAST_dt']  + pd.Timedelta(hours=OFFSET_HORAS)

# Duración en us y ms
df_nemea['flow_dur_us'] = (
    (df_nemea['TIME_LAST_dt'] - df_nemea['TIME_FIRST_dt']).dt.total_seconds() * 1_000_000
).round().astype('Int64')
df_nemea['flow_dur_ms'] = (df_nemea['flow_dur_us'] / 1000).round().astype('Int64')

print(f'Shape Nemea limpio: {df_nemea.shape}')
print(f'\nRango temporal Nemea (UTC+{OFFSET_HORAS}):')
print(f"  Min: {df_nemea['TIME_FIRST_utc2'].min()}")
print(f"  Max: {df_nemea['TIME_FIRST_utc2'].max()}")
print(f'\nRango CIC (comprobar alineamiento):')
print(f'  Min: {df_cic["Timestamp"].min()}')
print(f'  Max: {df_cic["Timestamp"].max()}')
print(f'\n⚠ Si los rangos están desplazados ~1h, cambia OFFSET_HORAS en CELDA 0 y reinicia.')

In [ ]:
# ============================================================
# CELDA 2.3 — KEY BASE Y MATCHING CIC ∩ NEMEA
# ============================================================

df_nemea['time_sec'] = df_nemea['TIME_FIRST_utc2'].dt.floor('s')

df_nemea['key_base_dir'] = list(zip(
    df_nemea[col_src_ip].astype(str), df_nemea[col_src_port].astype(str),
    df_nemea[col_dst_ip].astype(str), df_nemea[col_dst_port].astype(str),
    df_nemea[col_protocol].astype(str), df_nemea['time_sec'].astype(str)
))
df_nemea['key_base_inv'] = list(zip(
    df_nemea[col_dst_ip].astype(str), df_nemea[col_dst_port].astype(str),
    df_nemea[col_src_ip].astype(str), df_nemea[col_src_port].astype(str),
    df_nemea[col_protocol].astype(str), df_nemea['time_sec'].astype(str)
))
df_nemea['key_exact_dir'] = list(zip(
    df_nemea[col_src_ip].astype(str), df_nemea[col_src_port].astype(str),
    df_nemea[col_dst_ip].astype(str), df_nemea[col_dst_port].astype(str),
    df_nemea[col_protocol].astype(str), df_nemea['time_sec'].astype(str),
    df_nemea['flow_dur_ms'].astype(str)
))
df_nemea['key_exact_inv'] = list(zip(
    df_nemea[col_dst_ip].astype(str), df_nemea[col_dst_port].astype(str),
    df_nemea[col_src_ip].astype(str), df_nemea[col_src_port].astype(str),
    df_nemea[col_protocol].astype(str), df_nemea['time_sec'].astype(str),
    df_nemea['flow_dur_ms'].astype(str)
))

cic_keys_base  = set(df_cic['key_base'])
cic_keys_exact = set(df_cic['key_exact'])

nemea_base_dir  = set(df_nemea['key_base_dir'])
nemea_base_inv  = set(df_nemea['key_base_inv'])
nemea_exact_dir = set(df_nemea['key_exact_dir'])
nemea_exact_inv = set(df_nemea['key_exact_inv'])

match_nemea_base  = cic_keys_base  & (nemea_base_dir  | nemea_base_inv)
match_nemea_exact = cic_keys_exact & (nemea_exact_dir | nemea_exact_inv)

print('=' * 60)
print('MATCHING CIC ∩ NEMEA — DoS')
print('=' * 60)
print(f'  CIC total flujos:            {len(df_cic):,}')
print(f'  Nemea total flujos:          {len(df_nemea):,}')
print()
print(f'  Match BASE (sin duración):   {len(match_nemea_base):,}  ({round(len(match_nemea_base)/len(cic_keys_base)*100,2)}%)')
print(f'  Match EXACTO (con dur. ms):  {len(match_nemea_exact):,}  ({round(len(match_nemea_exact)/len(cic_keys_base)*100,2)}%)')

# Liberar sets temporales y columnas que ya no se necesitan
del nemea_base_dir, nemea_base_inv, nemea_exact_dir, nemea_exact_inv
df_nemea = df_nemea.drop(columns=['key_exact_dir', 'key_exact_inv'])
gc.collect()

# TSTAT

In [ ]:
# ============================================================
# CELDA 3.1 — CARGA DE TSTAT
# DoS tiene 4 CSVs → glob
# ============================================================

RUTA_TSTAT = Path('/home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosTstat/CSV/DoS')
print(f'Ruta Tstat: {RUTA_TSTAT}')
print(f'Existe:     {RUTA_TSTAT.exists()}')
print()

csv_files_tstat = sorted(RUTA_TSTAT.glob('*.csv'))
print(f'CSVs encontrados: {len(csv_files_tstat)}')

dfs_tstat = []
for f in csv_files_tstat:
    df_temp = pd.read_csv(f, low_memory=False)
    dfs_tstat.append(df_temp)
    print(f'  ✓ {f.name:40s} → {len(df_temp):>8,} filas')

df_tstat = pd.concat(dfs_tstat, ignore_index=True)
del dfs_tstat
gc.collect()

print()
print(f'Shape total Tstat: {df_tstat.shape}')
print(f'NaNs:              {df_tstat.isnull().sum().sum():,}')
print(f'\nDistribución por protocolo:')
print(df_tstat['protocolo'].value_counts().to_string())
print(f"\nFilas con first == 0: {(df_tstat['first'] == 0).sum():,}")
print(f"Filas con first > 0:  {(df_tstat['first'] > 0).sum():,}")

In [ ]:
# ============================================================
# CELDA 3.2 — LIMPIEZA Y SEPARACIÓN TCP/UDP TSTAT
# ============================================================

df_tcp_tstat = df_tstat[df_tstat['protocolo'] == 'TCP'].copy()
df_udp_tstat = df_tstat[df_tstat['protocolo'] == 'UDP'].copy()
print(f'TCP total: {len(df_tcp_tstat):,}  | UDP total: {len(df_udp_tstat):,}')
print()

# ── TCP
df_tcp_tstat['first'] = pd.to_numeric(df_tcp_tstat['first'], errors='coerce')
n_antes = len(df_tcp_tstat)
df_tcp_tstat = df_tcp_tstat[df_tcp_tstat['first'] > 0].copy()
print(f'TCP válidos (first>0): {len(df_tcp_tstat):,}  (eliminados: {n_antes-len(df_tcp_tstat):,})')

df_tcp_tstat['first_dt']    = pd.to_datetime(df_tcp_tstat['first'], unit='ms', errors='coerce')
df_tcp_tstat['first_utc2']  = df_tcp_tstat['first_dt'] + pd.Timedelta(hours=OFFSET_HORAS)
df_tcp_tstat['flow_dur_ms'] = df_tcp_tstat['durat'].round().astype('Int64')
df_tcp_tstat['flow_dur_us'] = (df_tcp_tstat['durat'] * 1000).round().astype('Int64')
df_tcp_tstat['proto_num']   = 6

# ── UDP
df_udp_tstat['c_first_abs'] = pd.to_numeric(df_udp_tstat['c_first_abs'], errors='coerce')
n_antes = len(df_udp_tstat)
df_udp_tstat = df_udp_tstat[df_udp_tstat['c_first_abs'] > 0].copy()
print(f'UDP válidos (c_first_abs>0): {len(df_udp_tstat):,}  (eliminados: {n_antes-len(df_udp_tstat):,})')

df_udp_tstat['first_dt']    = pd.to_datetime(df_udp_tstat['c_first_abs'], unit='ms', errors='coerce')
df_udp_tstat['first_utc2']  = df_udp_tstat['first_dt'] + pd.Timedelta(hours=OFFSET_HORAS)
df_udp_tstat['c_durat']     = pd.to_numeric(df_udp_tstat['c_durat'], errors='coerce').fillna(0)
df_udp_tstat['s_durat']     = pd.to_numeric(df_udp_tstat['s_durat'], errors='coerce').fillna(0)
df_udp_tstat['flow_dur_ms'] = df_udp_tstat[['c_durat','s_durat']].max(axis=1).round().astype('Int64')
df_udp_tstat['flow_dur_us'] = (df_udp_tstat['flow_dur_ms'] * 1000).round().astype('Int64')
df_udp_tstat['proto_num']   = 17

print(f"\nUDP con duración > 0: {(df_udp_tstat['flow_dur_ms'] > 0).sum():,}")

del df_tstat
gc.collect()

In [ ]:
# ============================================================
# CELDA 3.3 — BASE UNIFICADA TSTAT
# ============================================================

cols_base_tstat = ['c_ip','c_port','s_ip','s_port','proto_num',
                   'first_utc2','flow_dur_us','flow_dur_ms']

df_tstat_base = pd.concat([
    df_tcp_tstat[cols_base_tstat],
    df_udp_tstat[cols_base_tstat]
], ignore_index=True)

df_tstat_base['time_sec'] = df_tstat_base['first_utc2'].dt.floor('s')

print(f'Shape Tstat base: {df_tstat_base.shape}')
print(f"  TCP: {(df_tstat_base['proto_num']==6).sum():,}")
print(f"  UDP: {(df_tstat_base['proto_num']==17).sum():,}")
print(f'\nRango temporal Tstat (UTC+{OFFSET_HORAS}):')
print(f"  Min: {df_tstat_base['time_sec'].min()}")
print(f"  Max: {df_tstat_base['time_sec'].max()}")
print(f'\nRango CIC:')
print(f"  Min: {df_cic['Timestamp'].min()}")
print(f"  Max: {df_cic['Timestamp'].max()}")

In [ ]:
# ============================================================
# CELDA 3.4 — KEY BASE Y MATCHING CIC ∩ TSTAT
# ============================================================

df_tstat_base['key_base_dir'] = list(zip(
    df_tstat_base['c_ip'].astype(str), df_tstat_base['c_port'].astype(str),
    df_tstat_base['s_ip'].astype(str), df_tstat_base['s_port'].astype(str),
    df_tstat_base['proto_num'].astype(str), df_tstat_base['time_sec'].astype(str)
))
df_tstat_base['key_base_inv'] = list(zip(
    df_tstat_base['s_ip'].astype(str), df_tstat_base['s_port'].astype(str),
    df_tstat_base['c_ip'].astype(str), df_tstat_base['c_port'].astype(str),
    df_tstat_base['proto_num'].astype(str), df_tstat_base['time_sec'].astype(str)
))

df_tstat_dur = df_tstat_base[df_tstat_base['flow_dur_ms'] > 0].copy()
df_tstat_dur['key_exact_dir'] = list(zip(
    df_tstat_dur['c_ip'].astype(str), df_tstat_dur['c_port'].astype(str),
    df_tstat_dur['s_ip'].astype(str), df_tstat_dur['s_port'].astype(str),
    df_tstat_dur['proto_num'].astype(str), df_tstat_dur['time_sec'].astype(str),
    df_tstat_dur['flow_dur_ms'].astype(str)
))
df_tstat_dur['key_exact_inv'] = list(zip(
    df_tstat_dur['s_ip'].astype(str), df_tstat_dur['s_port'].astype(str),
    df_tstat_dur['c_ip'].astype(str), df_tstat_dur['c_port'].astype(str),
    df_tstat_dur['proto_num'].astype(str), df_tstat_dur['time_sec'].astype(str),
    df_tstat_dur['flow_dur_ms'].astype(str)
))

tstat_base_keys  = set(df_tstat_base['key_base_dir']) | set(df_tstat_base['key_base_inv'])
tstat_exact_keys = set(df_tstat_dur['key_exact_dir']) | set(df_tstat_dur['key_exact_inv'])

match_tstat_base  = cic_keys_base  & tstat_base_keys
match_tstat_exact = cic_keys_exact & tstat_exact_keys

print('=' * 60)
print('MATCHING CIC ∩ TSTAT — DoS')
print('=' * 60)
print(f'  Tstat total flujos:          {len(df_tstat_base):,}')
print(f'  Tstat con duración > 0:      {len(df_tstat_dur):,}')
print(f'  Tstat sin duración:          {len(df_tstat_base)-len(df_tstat_dur):,}')
print()
print(f'  Match BASE (sin duración):   {len(match_tstat_base):,}  ({round(len(match_tstat_base)/len(cic_keys_base)*100,2)}%)')
print(f'  Match EXACTO (con dur. ms):  {len(match_tstat_exact):,}  ({round(len(match_tstat_exact)/len(cic_keys_base)*100,2)}%)')

# Liberar sets temporales y dataframes intermedios
del tstat_base_keys, tstat_exact_keys, df_tcp_tstat, df_udp_tstat, df_tstat_dur
gc.collect()

# TSHARK

In [ ]:
# ===============================================
# CELDA 4.1 — CARGA DE TSHARK
# DoS tiene 4 CSVs → glob + sampling configurable
# ===============================================

RUTA_TSHARK = Path('/home/miguel/Escritorio/TFM/TFM_Miguel/ArchivosTshark_2/Csv/DoS')
print(f'Ruta Tshark: {RUTA_TSHARK}')
print(f'Existe:      {RUTA_TSHARK.exists()}')
print()

csv_files_tshark = sorted(RUTA_TSHARK.glob('*.csv'))
print(f'CSVs encontrados: {len(csv_files_tshark)}')
print(f'Sampling activo: N_SAMPLE_TSHARK = {N_SAMPLE_TSHARK:,} filas/CSV')
print()

dfs_tshark = []
for f in csv_files_tshark:
    df_temp = pd.read_csv(f, low_memory=False)
    if len(df_temp) > N_SAMPLE_TSHARK:
        df_temp = df_temp.sample(N_SAMPLE_TSHARK, random_state=42)
    df_temp['archivo_origen'] = f.stem
    dfs_tshark.append(df_temp)
    print(f'  ✓ {f.name:40s} → {len(df_temp):>8,} filas')

df_tshark = pd.concat(dfs_tshark, ignore_index=True)
del dfs_tshark
gc.collect()

print()
print(f'Shape Tshark: {df_tshark.shape}')
print(f'NaNs:         {df_tshark.isnull().sum().sum():,}')
print(f'\nDistribución por protocolo:')
print(df_tshark['proto'].value_counts().to_string())
print(f'\nDistribución por subtipo:')
print(df_tshark['archivo_origen'].value_counts().to_string())

In [ ]:
# ============================================================
# CELDA 4.2 — LIMPIEZA Y NORMALIZACIÓN TSHARK
# ============================================================

df_tshark['start_dt']    = pd.to_datetime(df_tshark['start_epoch'], unit='s', errors='coerce')
df_tshark['start_utc2']  = df_tshark['start_dt'] + pd.Timedelta(hours=OFFSET_HORAS)
df_tshark['flow_dur_us'] = (df_tshark['flow_duration_s'] * 1_000_000).round().astype('Int64')
df_tshark['flow_dur_ms'] = (df_tshark['flow_duration_s'] * 1_000).round().astype('Int64')
df_tshark['proto_num']   = pd.to_numeric(df_tshark['proto_num'], errors='coerce').fillna(0).astype(int)

n_antes = len(df_tshark)
df_tshark = df_tshark.dropna(subset=['src_ip','dst_ip']).reset_index(drop=True)
print(f'Filas eliminadas sin IP: {n_antes - len(df_tshark):,}')

print(f'\nShape Tshark limpio: {df_tshark.shape}')
print(f'\nRango temporal Tshark (UTC+{OFFSET_HORAS}):')
print(f"  Min: {df_tshark['start_utc2'].min()}")
print(f"  Max: {df_tshark['start_utc2'].max()}")
print(f'\nRango CIC:')
print(f"  Min: {df_cic['Timestamp'].min()}")
print(f"  Max: {df_cic['Timestamp'].max()}")

In [ ]:
# ============================================================
# CELDA 4.3 — KEY BASE Y MATCHING CIC ∩ TSHARK
# ============================================================

df_tshark['time_sec'] = df_tshark['start_utc2'].dt.floor('s')

df_tshark['key_base_dir'] = list(zip(
    df_tshark['src_ip'].astype(str), df_tshark['src_port'].astype(str),
    df_tshark['dst_ip'].astype(str), df_tshark['dst_port'].astype(str),
    df_tshark['proto_num'].astype(str), df_tshark['time_sec'].astype(str)
))
df_tshark['key_base_inv'] = list(zip(
    df_tshark['dst_ip'].astype(str), df_tshark['dst_port'].astype(str),
    df_tshark['src_ip'].astype(str), df_tshark['src_port'].astype(str),
    df_tshark['proto_num'].astype(str), df_tshark['time_sec'].astype(str)
))
df_tshark['key_exact_dir'] = list(zip(
    df_tshark['src_ip'].astype(str), df_tshark['src_port'].astype(str),
    df_tshark['dst_ip'].astype(str), df_tshark['dst_port'].astype(str),
    df_tshark['proto_num'].astype(str), df_tshark['time_sec'].astype(str),
    df_tshark['flow_dur_ms'].astype(str)
))
df_tshark['key_exact_inv'] = list(zip(
    df_tshark['dst_ip'].astype(str), df_tshark['dst_port'].astype(str),
    df_tshark['src_ip'].astype(str), df_tshark['src_port'].astype(str),
    df_tshark['proto_num'].astype(str), df_tshark['time_sec'].astype(str),
    df_tshark['flow_dur_ms'].astype(str)
))

tshark_base_keys  = set(df_tshark['key_base_dir']) | set(df_tshark['key_base_inv'])
tshark_exact_keys = set(df_tshark['key_exact_dir']) | set(df_tshark['key_exact_inv'])

match_tshark_base  = cic_keys_base  & tshark_base_keys
match_tshark_exact = cic_keys_exact & tshark_exact_keys

print('=' * 60)
print('MATCHING CIC ∩ TSHARK — DoS')
print('=' * 60)
print(f'  Tshark total flujos:         {len(df_tshark):,}')
print()
print(f'  Match BASE (sin duración):   {len(match_tshark_base):,}  ({round(len(match_tshark_base)/len(cic_keys_base)*100,2)}%)')
print(f'  Match EXACTO (con dur. ms):  {len(match_tshark_exact):,}  ({round(len(match_tshark_exact)/len(cic_keys_base)*100,2)}%)')
print(f'  Diferencia:                  {len(match_tshark_base)-len(match_tshark_exact):,}')

# Liberar sets temporales y columnas key_exact
del tshark_base_keys, tshark_exact_keys
df_tshark = df_tshark.drop(columns=['key_exact_dir', 'key_exact_inv'])
gc.collect()

# Intersección las 4

In [ ]:
# ============================================================
# CELDA 5.1 — INTERSECCIÓN 4 HERRAMIENTAS
# ============================================================

base_4  = match_nemea_base  & match_tstat_base  & match_tshark_base
exact_4 = match_nemea_exact & match_tstat_exact & match_tshark_exact

print('=' * 60)
print('INTERSECCIÓN 4 HERRAMIENTAS — DoS')
print('=' * 60)
print(f'  CIC total claves:              {len(cic_keys_base):>8,}')
print(f'  CIC ∩ Nemea  (base):           {len(match_nemea_base):>8,}  ({round(len(match_nemea_base)/len(cic_keys_base)*100,2)}%)')
print(f'  CIC ∩ Tstat  (base):           {len(match_tstat_base):>8,}  ({round(len(match_tstat_base)/len(cic_keys_base)*100,2)}%)')
print(f'  CIC ∩ Tshark (base):           {len(match_tshark_base):>8,}  ({round(len(match_tshark_base)/len(cic_keys_base)*100,2)}%)')
print()
print(f'  COMUNES 4 (base, sin dur.):    {len(base_4):>8,}  ({round(len(base_4)/len(cic_keys_base)*100,2)}%)')
print(f'  COMUNES 4 (exacto, con dur.):  {len(exact_4):>8,}  ({round(len(exact_4)/len(cic_keys_base)*100,2)}%)')
print()
print(f'  Flujos extra al quitar dur.:   {len(base_4)-len(exact_4):>8,}')
print()
if len(base_4) > 3481:
    print(f'  → ✓ Supera la minoritaria actual (Web: 3.481)')
else:
    print(f'  → ⚠ NO supera la minoritaria (Web: 3.481). Revisar OFFSET_HORAS o sampling.')

In [ ]:
# ============================================================
# CELDA 5.2 — GRÁFICA RESUMEN
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('DoS — Resumen del matching entre herramientas',
             fontsize=13, fontweight='bold')

ax1 = axes[0]
herramientas = ['CIC', 'Nemea', 'Tstat', 'Tshark']
totales = [len(df_cic), len(df_nemea), len(df_tstat_base), len(df_tshark)]
base    = [len(cic_keys_base), len(match_nemea_base), len(match_tstat_base), len(match_tshark_base)]
exactos = [len(cic_keys_base), len(match_nemea_exact), len(match_tstat_exact), len(match_tshark_exact)]

x, width = np.arange(len(herramientas)), 0.25
b1 = ax1.bar(x - width, totales, width, label='Total flujos',            color='steelblue')
b2 = ax1.bar(x,         base,    width, label='Match base (sin dur.)',   color='darkorange')
b3 = ax1.bar(x + width, exactos, width, label='Match exacto (con dur.)', color='green')
ax1.set_title('Flujos por herramienta')
ax1.set_ylabel('Nº flujos')
ax1.set_xticks(x)
ax1.set_xticklabels(herramientas)
ax1.legend(fontsize=9)
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2, h,
                 f'{h:,}', ha='center', va='bottom', fontsize=7, rotation=45)

ax2 = axes[1]
labels2 = ['CIC\ntotal','CIC∩Nemea','CIC∩Tstat','CIC∩Tshark',
           'Comunes 4\n(base)','Comunes 4\n(exacto)']
values2 = [len(cic_keys_base), len(match_nemea_base), len(match_tstat_base),
           len(match_tshark_base), len(base_4), len(exact_4)]
colors2 = ['steelblue','darkorange','green','purple','crimson','darkred']
bars2 = ax2.bar(labels2, values2, color=colors2)
ax2.set_title('Embudo: sin duración vs con duración')
ax2.set_ylabel('Nº flujos')
for bar, val in zip(bars2, values2):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{val:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELDA 5.3 — EXTRACCIÓN FINAL: 4 CSVs CON TODAS LAS COLUMNAS
# ============================================================

OUTPUT_DIR = Path('/home/miguel/Escritorio/TFM/TFM_Miguel/MuestrasComunes/DoS')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLS_AUX_CIC = ['archivo_origen','time_sec','flow_dur_ms','key_base','key_exact']

COLS_AUX_NEMEA = ['TIME_FIRST_limpio','TIME_LAST_limpio',
                  'TIME_FIRST_dt','TIME_LAST_dt',
                  'TIME_FIRST_utc2','TIME_LAST_utc2',
                  'time_sec','flow_dur_us','flow_dur_ms',
                  'key_base_dir','key_base_inv','_key']

COLS_AUX_TSTAT = ['first_utc2','flow_dur_us','flow_dur_ms',
                  'time_sec','key_base_dir','key_base_inv',
                  '_key','proto_num','first_dt']

COLS_AUX_TSHARK = ['start_dt','start_utc2','flow_dur_us','flow_dur_ms',
                   'time_sec','key_base_dir','key_base_inv',
                   'proto_num','archivo_origen','_key']

# ── NEMEA
df_nemea['_key'] = None
df_nemea.loc[df_nemea['key_base_dir'].isin(base_4), '_key'] = \
    df_nemea.loc[df_nemea['key_base_dir'].isin(base_4), 'key_base_dir']
df_nemea.loc[df_nemea['key_base_inv'].isin(base_4) & df_nemea['_key'].isna(), '_key'] = \
    df_nemea.loc[df_nemea['key_base_inv'].isin(base_4) & df_nemea['_key'].isna(), 'key_base_inv']

df_nemea_4 = df_nemea[df_nemea['_key'].isin(base_4)].copy()
df_nemea_4 = df_nemea_4.drop_duplicates(subset=['_key'], keep='first')
claves_nemea_final = set(df_nemea_4['_key'])
df_nemea_4 = df_nemea_4.drop(columns=COLS_AUX_NEMEA, errors='ignore').reset_index(drop=True)
print(f'Nemea: {len(df_nemea_4):,} filas')

# ── TSTAT
df_tstat_base['_key'] = None
df_tstat_base.loc[df_tstat_base['key_base_dir'].isin(base_4), '_key'] = \
    df_tstat_base.loc[df_tstat_base['key_base_dir'].isin(base_4), 'key_base_dir']
df_tstat_base.loc[df_tstat_base['key_base_inv'].isin(base_4) & df_tstat_base['_key'].isna(), '_key'] = \
    df_tstat_base.loc[df_tstat_base['key_base_inv'].isin(base_4) & df_tstat_base['_key'].isna(), 'key_base_inv']

df_tstat_4 = df_tstat_base[df_tstat_base['_key'].isin(claves_nemea_final)].copy()
df_tstat_4 = df_tstat_4.drop_duplicates(subset=['_key'], keep='first')
claves_tstat_final = set(df_tstat_4['_key'])
df_tstat_4 = df_tstat_4.drop(columns=COLS_AUX_TSTAT, errors='ignore').reset_index(drop=True)
print(f'Tstat: {len(df_tstat_4):,} filas')

# ── TSHARK
df_tshark['_key'] = None
df_tshark.loc[df_tshark['key_base_dir'].isin(base_4), '_key'] = \
    df_tshark.loc[df_tshark['key_base_dir'].isin(base_4), 'key_base_dir']
df_tshark.loc[df_tshark['key_base_inv'].isin(base_4) & df_tshark['_key'].isna(), '_key'] = \
    df_tshark.loc[df_tshark['key_base_inv'].isin(base_4) & df_tshark['_key'].isna(), 'key_base_inv']

df_tshark_4 = df_tshark[df_tshark['_key'].isin(claves_tstat_final)].copy()
df_tshark_4 = df_tshark_4.drop_duplicates(subset=['_key'], keep='first')
claves_tshark_final = set(df_tshark_4['_key'])
df_tshark_4 = df_tshark_4.drop(columns=COLS_AUX_TSHARK, errors='ignore').reset_index(drop=True)
print(f'Tshark: {len(df_tshark_4):,} filas')

# ── CIC (referencia final)
df_cic_4 = df_cic[df_cic['key_base'].isin(claves_tshark_final)].copy()
df_cic_4 = df_cic_4.drop_duplicates(subset=['key_base'], keep='first')
df_cic_4 = df_cic_4.drop(columns=COLS_AUX_CIC, errors='ignore').reset_index(drop=True)
print(f'CIC:   {len(df_cic_4):,} filas')

# Re-filtrar Nemea/Tstat con las claves finales de Tshark para garantizar coincidencia
df_nemea_4 = df_nemea_4[df_nemea_4.index.isin(
    df_nemea[df_nemea['_key'].isin(claves_tshark_final)].drop_duplicates(subset=['_key']).index
)].reset_index(drop=True) if '_key' in df_nemea.columns else df_nemea_4

print()
print(f'Claves comunes reales (en las 4): {len(claves_tshark_final):,}')

# Guardar
df_cic_4.to_csv(   OUTPUT_DIR / 'CIC_comunes.csv',    index=False)
df_nemea_4.to_csv( OUTPUT_DIR / 'Nemea_comunes.csv',  index=False)
df_tshark_4.to_csv(OUTPUT_DIR / 'Tshark_comunes.csv', index=False)
df_tstat_4.to_csv( OUTPUT_DIR / 'Tstat_comunes.csv',  index=False)

print()
print('=' * 60)
print('RESULTADO FINAL — DoS')
print('=' * 60)
print(f'  CIC:    {len(df_cic_4):>6,} filas x {df_cic_4.shape[1]:>3} columnas')
print(f'  Nemea:  {len(df_nemea_4):>6,} filas x {df_nemea_4.shape[1]:>3} columnas')
print(f'  Tstat:  {len(df_tstat_4):>6,} filas x {df_tstat_4.shape[1]:>3} columnas')
print(f'  Tshark: {len(df_tshark_4):>6,} filas x {df_tshark_4.shape[1]:>3} columnas')
print(f'\n  Guardados en: {OUTPUT_DIR}')

if len(df_cic_4) == len(df_nemea_4) == len(df_tstat_4) == len(df_tshark_4):
    print(f'\n  ✓ VERIFICACIÓN CORRECTA — los 4 CSVs tienen {len(df_cic_4):,} flujos')
else:
    print(f'\n  ✗ Algún CSV no coincide → revisar')

In [ ]:
# ============================================================
# CELDA 5.4 — ESTANDARIZAR ETIQUETA A 'DoS' EN LOS 4 CSVs
# Cada herramienta usa nombre de columna distinto y puede tener
# subtipos (DoS-TCP, DoS-UDP, DoS-SYN, DoS-HTTP). Se unifican
# todos a la clase 'DoS' para el clasificador ML.
# ============================================================

LABEL_MAP = {
    'CIC':    (df_cic_4,    'Label'),
    'Nemea':  (df_nemea_4,  col_etiqueta_nemea),
    'Tshark': (df_tshark_4, 'category'),
    'Tstat':  (df_tstat_4,  'etiqueta'),
}

for nombre, (df, col) in LABEL_MAP.items():
    if col in df.columns:
        antes = df[col].value_counts().to_dict()
        df[col] = 'DoS'
        print(f'{nombre:8s} | {col:20s} → antes: {antes}')
        print(f'{"":8s} |                       → ahora: {df[col].unique().tolist()}')
    else:
        print(f'  ⚠ {nombre}: columna \'{col}\' no encontrada')

# Sobreescribir con etiquetas actualizadas
df_cic_4.to_csv(   OUTPUT_DIR / 'CIC_comunes.csv',    index=False)
df_nemea_4.to_csv( OUTPUT_DIR / 'Nemea_comunes.csv',  index=False)
df_tshark_4.to_csv(OUTPUT_DIR / 'Tshark_comunes.csv', index=False)
df_tstat_4.to_csv( OUTPUT_DIR / 'Tstat_comunes.csv',  index=False)

print()
print('✓ Etiquetas actualizadas y CSVs sobreescritos')